# %% [markdown]
# # 🔍 EDA - Household Electric Power Consumption
# ## Analyse exploratoire des données

In [1]:
# %%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [ ]:
# Configuration des plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

In [ ]:
# %%
# Chargement des données nettoyées
df = pd.read_parquet("../data/processed/cleaned_data.parquet")
print(f"Shape: {df.shape}")
print(f"\nPlage temporelle: {df.index.min()} → {df.index.max()}")
print(f"\nColonnes: {list(df.columns)}")

In [ ]:
# %%
# Résumé statistique
df.describe()

In [ ]:
# %%
# Distribution des valeurs manquantes (avant cleaning)
raw_df = pd.read_parquet("../data/processed/raw_loaded.parquet")
missing = raw_df.isnull().sum()
missing_pct = (missing / len(raw_df)) * 100
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
missing_df = missing_df[missing_df['count'] > 0].sort_values('pct', ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(missing_df.index, missing_df['pct'], color='coral')
ax.set_ylabel('% Valeurs manquantes')
ax.set_title('Valeurs manquantes par colonne (avant nettoyage)')
ax.set_ylim(0, missing_df['pct'].max() * 1.2)
for bar, pct in zip(bars, missing_df['pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, 
            f'{pct:.2f}%', ha='center', va='bottom')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../notebooks/figures/missing_values.png", dpi=150, bbox_inches='tight')
plt.show()

# %% [markdown]
# ## 📈 Série temporelle globale

In [ ]:
# %%
# Resampling horaire pour visualisation
df_hourly = df.resample('H').mean()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# Global active power
axes[0].plot(df_hourly.index, df_hourly['Global_active_power'], 
             color='steelblue', alpha=0.8, linewidth=0.5)
axes[0].set_ylabel('Global Active Power (kW)')
axes[0].set_title('Consommation électrique globale (horaire)')
axes[0].axhline(y=df_hourly['Global_active_power'].mean(), 
                color='red', linestyle='--', alpha=0.7, label='Moyenne')

# Sub-meterings
axes[1].plot(df_hourly.index, df_hourly['Sub_metering_1'], 
             label='Cuisine', alpha=0.8, linewidth=0.5)
axes[1].plot(df_hourly.index, df_hourly['Sub_metering_2'], 
             label='Buanderie', alpha=0.8, linewidth=0.5)
axes[1].plot(df_hourly.index, df_hourly['Sub_metering_3'], 
             label='Chauffe-eau/Clim', alpha=0.8, linewidth=0.5)
axes[1].set_ylabel('Sub-metering (Wh)')
axes[1].set_title('Consommation par zone (horaire)')
axes[1].legend()
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.savefig("../notebooks/figures/time_series_overview.png", dpi=150, bbox_inches='tight')
plt.show()

# %% [markdown]
# ## 🗓️ Patterns saisonniers

In [ ]:
# %%
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Pattern horaire
hourly_pattern = df.groupby('hour')['Global_active_power'].mean()
axes[0,0].plot(hourly_pattern.index, hourly_pattern.values, marker='o', color='steelblue')
axes[0,0].set_title('Pattern horaire moyen')
axes[0,0].set_xlabel('Heure')
axes[0,0].set_ylabel('Global Active Power (kW)')
axes[0,0].set_xticks(range(0, 24, 2))
# Pattern jour de semaine
dow_pattern = df.groupby('day_of_week')['Global_active_power'].mean()
dow_labels = ['Lun', 'Mar', 'Mer', 'Jeu', 'Ven', 'Sam', 'Dim']
axes[0,1].bar(range(7), dow_pattern.values, color='coral')
axes[0,1].set_title('Pattern jour de semaine')
axes[0,1].set_xlabel('Jour')
axes[0,1].set_ylabel('Global Active Power (kW)')
axes[0,1].set_xticks(range(7))
axes[0,1].set_xticklabels(dow_labels)

In [ ]:
# Pattern mensuel
monthly_pattern = df.groupby('month')['Global_active_power'].mean()
axes[1,0].plot(monthly_pattern.index, monthly_pattern.values, marker='o', color='green')
axes[1,0].set_title('Pattern mensuel')
axes[1,0].set_xlabel('Mois')
axes[1,0].set_ylabel('Global Active Power (kW)')
axes[1,0].set_xticks(range(1, 13))

In [ ]:
# Weekend vs Semaine
weekend_pattern = df.groupby(['hour', 'is_weekend'])['Global_active_power'].mean().unstack()
axes[1,1].plot(weekend_pattern.index, weekend_pattern[0], label='Semaine', marker='o')
axes[1,1].plot(weekend_pattern.index, weekend_pattern[1], label='Weekend', marker='s')
axes[1,1].set_title('Semaine vs Weekend')
axes[1,1].set_xlabel('Heure')
axes[1,1].set_ylabel('Global Active Power (kW)')
axes[1,1].legend()

plt.tight_layout()
plt.savefig("../notebooks/figures/seasonal_patterns.png", dpi=150, bbox_inches='tight')
plt.show()

# %% [markdown]
# ## 🔥 Heatmap heure × jour de semaine

In [ ]:
# %%
pivot = df.pivot_table(values='Global_active_power', 
                       index='hour', 
                       columns='day_of_week', 
                       aggfunc='mean')

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(pivot, cmap='YlOrRd', annot=False, fmt='.2f', 
            xticklabels=dow_labels, ax=ax)
ax.set_title('Heatmap consommation moyenne (Heure × Jour)')
ax.set_xlabel('Jour de la semaine')
ax.set_ylabel('Heure')
plt.tight_layout()
plt.savefig("../notebooks/figures/heatmap_seasonal.png", dpi=150, bbox_inches='tight')
plt.show()

# %% [markdown]
# ## 📊 Corrélations

In [ ]:
# %%
# Matrice de corrélation
numeric_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 
                'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 
                'Sub_metering_3', 'Unmeasured_consumption']

corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', 
            cmap='RdBu_r', center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Matrice de corrélation')
plt.tight_layout()
plt.savefig("../notebooks/figures/correlation_matrix.png", dpi=150, bbox_inches='tight')
plt.show()

# %% [markdown]
# ## 📉 Distribution des variables

In [ ]:
# %%
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

cols_to_plot = ['Global_active_power', 'Voltage', 'Global_intensity',
                'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']

for ax, col in zip(axes.flatten(), cols_to_plot):
    ax.hist(df[col].dropna(), bins=100, color='steelblue', alpha=0.7, edgecolor='black')
    ax.set_title(f'Distribution de {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Fréquence')
plt.tight_layout()
plt.savefig("../notebooks/figures/distributions.png", dpi=150, bbox_inches='tight')
plt.show()

# %% [markdown]
# ## 💡 Insights clés

In [ ]:
# %%
print("=" * 60)
print("📊 INSIGHTS CLÉS DE L'EDA")
print("=" * 60)

print(f"\n1. TAILLE DU DATASET")
print(f"   • {len(df):,} lignes ({len(df)/1e6:.2f}M)")
print(f"   • {df.shape[1]} colonnes")
print(f"   • Période : {df.index.min().date()} → {df.index.max().date()}")

print(f"\n2. VALEURS MANQUANTES")
print(f"   • ~{(raw_df.isnull().sum().sum() / raw_df.size * 100):.2f}% des valeurs")
print(f"   • Principalement sur les sub-meterings")
print(f"\n3. CONSOMMATION GLOBALE")
print(f"   • Moyenne : {df['Global_active_power'].mean():.3f} kW")
print(f"   • Max : {df['Global_active_power'].max():.3f} kW")
print(f"   • Min : {df['Global_active_power'].min():.3f} kW")

print(f"\n4. PATTERNS TEMPORELS")
print(f"   • Heure creuse : {hourly_pattern.idxmin()}h ({hourly_pattern.min():.3f} kW)")
print(f"   • Heure pleine : {hourly_pattern.idxmax()}h ({hourly_pattern.max():.3f} kW)")
print(f"   • Weekend vs Semaine : ratio = {dow_pattern.loc[5:].mean()/dow_pattern.loc[:4].mean():.3f}")
print(f"\n5. SUB-METERINGS")
total_sub = df[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].sum()
print(f"   • Cuisine : {total_sub['Sub_metering_1']/total_sub.sum()*100:.1f}%")
print(f"   • Buanderie : {total_sub['Sub_metering_2']/total_sub.sum()*100:.1f}%")
print(f"   • Chauffe-eau : {total_sub['Sub_metering_3']/total_sub.sum()*100:.1f}%")

print(f"\n6. CONSOMMATION NON MESURÉE")
unmeasured_pct = (df['Unmeasured_consumption'].sum() / 
                  (df['Global_active_power'].sum() * 1000 / 60) * 100)
print(f"   • ~{unmeasured_pct:.1f}% de la consommation totale")

print("\n" + "=" * 60)